# DAY 2B | The Same RAG, Now in a Vector Database (Pinecone)

**Decoding Data Science | AI Accelerator Bootcamp (CPD UK)**

| | |
|---|---|
| **Time** | about 30 to 40 minutes (after the Day 2 notebook) |
| **Goal** | Show the whole journey: first the search index lives in **memory**, then in a real **vector database** |
| **You will build** | The same DecoBot policy search, stored in Pinecone, that survives a restart |
| **Files needed** | `DDS_Leave_Policy_v2.pdf` |
| **Keys needed** | `GOOGLE_API_KEY` and `PINECONE_API_KEY`, both in Colab Secrets |
| **Tools** | Google Colab + Gemini + LlamaIndex + Pinecone |

---

### The journey

| | Day 2 notebook | This notebook (2B) |
|---|---|---|
| Where the vectors live | In the notebook's memory | In Pinecone (a database in the cloud) |
| Close the notebook | Everything is lost, we embed again | Everything is still there |
| Embedding cost | Paid again on every run | Paid once, reused |
| Fit for | Learning and testing | A real chatbot that many people use |

```
PDF -> sections -> embeddings -> [ memory ]            -> search -> Gemini answers   (Part B)
PDF -> sections -> embeddings -> [ Pinecone database ] -> search -> Gemini answers   (Parts C, D, E)
```

## Part 0 | Setup (10 min)

In [ ]:
%pip install -q llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-vector-stores-pinecone pinecone pypdf

import importlib.metadata as md
for pkg in ["llama-index-core", "llama-index-llms-google-genai", "llama-index-embeddings-google-genai",
            "llama-index-vector-stores-pinecone", "pinecone", "pypdf"]:
    print(f"{pkg:40s}", md.version(pkg))

Upload the policy PDF (`DDS_Leave_Policy_v2.pdf`) when the file picker opens.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

> **Before you run the next cell:** click the key icon on the left of Colab (Secrets). Add **two** secrets and switch on "Notebook access" for both:
> 1. `GOOGLE_API_KEY` (from Google AI Studio)
> 2. `PINECONE_API_KEY` (from the Pinecone console, free Starter plan is enough)
>
> Never paste a key into a cell.

In [ ]:
import os
from google.colab import userdata

for name in ["GOOGLE_API_KEY", "PINECONE_API_KEY"]:
    try:
        os.environ[name] = userdata.get(name)
    except Exception as error:
        raise RuntimeError(
            f"Colab secret {name} not found. Click the key icon on the left of Colab, add a secret named {name}, "
            "switch on Notebook access, then run this cell again."
        ) from error
    print(name, "loaded:", bool(os.environ[name]))

In [ ]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

MODEL = "gemini-3.8-flash"                # change only these two lines if Google renames a model
EMBED_MODEL = "gemini-embedding-001"

Settings.llm = GoogleGenAI(model=MODEL, temperature=0.1)
Settings.embed_model = GoogleGenAIEmbedding(model_name=EMBED_MODEL, embed_batch_size=50)

print(Settings.llm.complete("Say hello to the DDS team in one short sentence.").text)

---
## Part A | The same chunks as Day 2 (5 min)

We reuse the winner of the Day 2 chunking lab: **one chunk per numbered section**, with the section title kept as metadata.
Nothing changes here. Only *where we store the chunks* changes in the next parts.

In [ ]:
import re
from pypdf import PdfReader
from llama_index.core.schema import TextNode

pages = [page.extract_text() for page in PdfReader("DDS_Leave_Policy_v2.pdf").pages]
full_text = "\n".join(pages).replace("\x7f", "-")          # clean PDF bullet characters

nodes = []
for part in re.split(r"(?m)^(?=\d{1,2}\.\s+[A-Z])", full_text):
    part = part.strip()
    if part:
        nodes.append(TextNode(text=part, metadata={"section": part.splitlines()[0]}))

print("Pages:", len(pages), "| Chunks:", len(nodes))
for n in nodes[:5]:
    print(" -", n.metadata["section"])

---
## Part B | Version 1: the index lives in memory (5 min)

This is exactly what the Day 2 notebook did. Watch two things: how long the embedding takes, and the word *memory*.

In [ ]:
import time
from llama_index.core import VectorStoreIndex, PromptTemplate

qa_prompt = PromptTemplate(
    "You are DecoBot, the leave assistant for Decoding Data Science (DDS).\n"
    "Answer ONLY from the policy excerpts below. Quote numbers exactly and mention the section.\n"
    "If the answer is not in the excerpts, reply exactly: "
    "'I could not find that in the DDS leave policy. Please contact People & Culture.'\n"
    "Keep the answer to 4 short lines.\n"
    "---------------------\n{context_str}\n---------------------\n"
    "Question: {query_str}\nAnswer: "
)

start = time.perf_counter()
memory_index = VectorStoreIndex(nodes)                       # embeds every chunk, keeps vectors in RAM
print(f"Built in memory in {time.perf_counter() - start:.1f} seconds")

memory_engine = memory_index.as_query_engine(similarity_top_k=3, text_qa_template=qa_prompt)
print(memory_engine.query("How much notice do I need to book 12 working days of annual leave?"))

**Checkpoint.** It works. But those vectors exist only while this notebook is running.
Restart the runtime and they are gone: every chunk would be embedded (and paid for) again.
A chatbot that many people use cannot work like that.

---
## Part C | Version 2: store the vectors in Pinecone (15 min)

**Pinecone** is a vector database: a service that stores vectors and finds the closest ones fast.
Three words to know:

| Word | Meaning |
|---|---|
| **Index** | One collection of vectors, like a table |
| **Dimension** | How many numbers each vector has. It must match the embedding model |
| **Metric** | How closeness is measured. We use `cosine` |

In [ ]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "dds-leave-policy"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# the dimension must match the embedding model, so we measure it instead of guessing
dimension = len(Settings.embed_model.get_text_embedding("dimension check"))
print("Embedding dimension:", dimension)

if not pc.has_index(INDEX_NAME):
    pc.create_index(name=INDEX_NAME, dimension=dimension, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    print("Created index:", INDEX_NAME)
else:
    print("Index already exists:", INDEX_NAME)

while not pc.describe_index(INDEX_NAME).status.ready:
    time.sleep(2)
print("Index is ready.")

Now we send the chunks to Pinecone. LlamaIndex embeds each chunk with Gemini and uploads the vector together with its text and section title.

In [ ]:
from llama_index.core import StorageContext
from llama_index.vector_stores.pinecone import PineconeVectorStore

pinecone_index = pc.Index(INDEX_NAME)
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

start = time.perf_counter()
pinecone_idx = VectorStoreIndex(nodes, storage_context=storage_context)
print(f"Uploaded to Pinecone in {time.perf_counter() - start:.1f} seconds")

time.sleep(5)                                                # give Pinecone a moment to finish indexing
print(pinecone_index.describe_index_stats())

**Checkpoint.** `total_vector_count` should equal the number of chunks from Part A.
Open the Pinecone console in your browser, click the index `dds-leave-policy` and look at the records: those are our policy sections.

---
## Part D | Ask DecoBot through Pinecone (10 min)

Same questions, same answer prompt. Only the storage changed.

In [ ]:
pinecone_engine = pinecone_idx.as_query_engine(similarity_top_k=3, text_qa_template=qa_prompt)

def ask_policy(question, engine=pinecone_engine):
    response = engine.query(question)
    print("QUESTION:", question)
    print("ANSWER  :", str(response).strip())
    sections = sorted({n.node.metadata.get("section", "?") for n in response.source_nodes})
    print("SOURCES :", " | ".join(sections))
    print()

ask_policy("How much notice do I need to book 12 working days of annual leave?")
ask_policy("What is the maximum number of days I can carry over?")
ask_policy("Do I need a medical certificate for 3 sick days?")

Now the two trap questions from Day 2. A good bot must refuse the first and cannot answer the second from a PDF.

In [ ]:
ask_policy("What is the office dress code?")                 # not in the policy: must say it could not find it
ask_policy("How many leave days do I have left?")            # a personal number: lives in the database (Day 3)

---
## Part E | The proof: restart and reconnect (5 min)

This is the reason for a vector database. We pretend the notebook was closed and opened again tomorrow:
we **do not read the PDF and do not embed anything**. We only connect to the index that already exists.

In [ ]:
del pinecone_idx, pinecone_engine                           # forget everything we built in this notebook

start = time.perf_counter()
reconnected_store = PineconeVectorStore(pinecone_index=pc.Index(INDEX_NAME))
reconnected_index = VectorStoreIndex.from_vector_store(reconnected_store)     # no embedding of chunks
print(f"Reconnected in {time.perf_counter() - start:.1f} seconds (nothing was embedded)")

ask_policy("How long is maternity leave and is it paid?", engine=reconnected_index.as_query_engine(
    similarity_top_k=3, text_qa_template=qa_prompt))

**Checkpoint.** DecoBot answers from the policy without reading the PDF again. That is what lets a website serve many users:
the website connects to Pinecone, and the policy is embedded once.

---
## Journey summary

| Step | Memory (Part B) | Pinecone (Parts C to E) |
|---|---|---|
| Build | Embeds every run | Embeds once |
| After a restart | Lost | Still there |
| Many users | Each copy re-embeds | All connect to one index |
| Update the policy | Re-run the notebook | Re-upload changed chunks |
| Extra cost | None | A Pinecone account (free Starter plan is enough here) |

### Clean up (optional)

Free Pinecone plans allow only a few indexes. When the class is over you can delete the demo index.
Leave this cell commented out if you want to keep it for Day 3 or for a website.

In [ ]:
# pc.delete_index(INDEX_NAME)
# print("Deleted index:", INDEX_NAME)

### Self-check
1. Why did we measure the embedding dimension instead of typing a number?
2. What was lost in Part B when the runtime restarted, and what was not lost in Part E?
3. Which two secrets does this notebook need, and where do they live?

### Next
Day 3 adds the employee database, three tools and an agent. The policy search tool can use either version you built today.